In [ ]:
import scanpy as sc
import pathlib
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import yaml
import seaborn as sns

In [ ]:
plot_dir = pathlib.Path("plots/sup_figure3")
plot_dir.mkdir(exist_ok=True)

In [ ]:
figure_config = yaml.safe_load(open("figures_config.yml", "r"))
matplotlib.rcParams.update(figure_config["font_config"])

In [ ]:
dataset_map = {
    "neftel_ss2_all_0": "Neftel, SS2 (4)",
    "neftel_10x_all_0": "Neftel, 10X (4)",
    "kim_lung_all_0": "Kim (3)",
    "ji_skin_all_0": "Ji (3)",
    "wu_subtype_all_0": "Wu (4)",
    "wu_gm_all_0": "Wu (6)",
    "rms_all_all_0": "RMS (6)",
}

In [ ]:
import matplotlib.gridspec as gridspec

# A4 size in inches (width x height)
A4_WIDTH = 8.27 - 0.25
A4_HEIGHT = 11.69 - 1.0

# Create figure with A4 size
fig = plt.figure(figsize=(2 * A4_WIDTH, 2 * A4_HEIGHT))
axes = {}
gs = gridspec.GridSpec(7, 2, figure=fig)

axes["neftel_ss2_all_0.h5ad"] = fig.add_subplot(gs[0, :])

axes["neftel_10x_all_0.h5ad"] = fig.add_subplot(gs[1, 0])
axes["ji_skin_all_0.h5ad"] = fig.add_subplot(gs[1, 1])

axes["kim_lung_all_0.h5ad"] = fig.add_subplot(gs[2, 0])

axes["wu_gm_all_0.h5ad"] = fig.add_subplot(gs[3, :])

axes["wu_subtype_all_0.h5ad"] = fig.add_subplot(gs[4, :])

axes["rms_all_all_0.h5ad_0"] = fig.add_subplot(gs[5, :])
axes["rms_all_all_0.h5ad_1"] = fig.add_subplot(gs[6, :])


for data_path in pathlib.Path(
    "../results/integration/umap/data/preprocessed"
).iterdir():
    if data_path.name.endswith(".rds"):
        continue

    if data_path.stem not in dataset_map.keys():
        continue
    if data_path.name == "rms_all_all_0.h5ad":
        obs = sc.read_h5ad(data_path).obs
        obs = obs[obs["phase"] == "G1"]
        for i in range(2):
            ax = axes[data_path.name + f"_{i}"]
            idx = slice(0, 39, 1) if i == 0 else slice(39, 77, 1)
            samples = obs["sample"].unique()[idx]

            pd.crosstab(
                obs["sample"][obs["sample"].isin(samples)],
                obs["celltype"][obs["sample"].isin(samples)],
                normalize="index",
            ).plot(kind="bar", stacked=True, ax=ax)

            sns.move_legend(
                ax,
                loc="center left",
                bbox_to_anchor=(1, 0.5),
                frameon=False,
                title="Cell state",
            )
            ax.set_xlabel("Sample")
            ax.set_ylabel("Proportion")
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)
            ax.set_title(dataset_map[data_path.stem])
            ax.set_yticks([0, 0.5, 1])

    else:
        obs = sc.read_h5ad(data_path).obs
        obs = obs[obs["phase"] == "G1"]

        ax = axes[data_path.name]
        pd.crosstab(obs["sample"], obs["celltype"], normalize="index").plot(
            kind="bar", stacked=True, ax=ax
        )
        sns.move_legend(
            ax,
            loc="center left",
            bbox_to_anchor=(1, 0.5),
            frameon=False,
            title="Cell state",
        )
        ax.set_xlabel("Sample")
        ax.set_ylabel("Proportion")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.set_title(dataset_map[data_path.stem])
        ax.set_yticks([0, 0.5, 1])
fig.tight_layout()
fig.savefig(plot_dir / "proportion.svg", bbox_inches="tight")

plt.show()

In [ ]:
idx

In [ ]:
obs["sample"][idx]

In [ ]:
obs["celltype"][idx]